# TrustShift-IDS — Revision Round 1 (post hoc reviewer-driven analyses)
All analyses in this notebook are **post hoc** (specified after the frozen V2 results were known).
Nothing here modifies any frozen V2 artifact: every output is written to `V2_adaptive_reliability/revision_R1/`.
Run cells in order: R0 → R1 → R2 → R3 → R4 → R5. A high-RAM runtime is enough; the GPU is not used
(XGBoost inference stays on CPU so that predictions are bit-comparable with the frozen results).

### CELL R0 — mount, provenance gate, shared helpers

In [ ]:
# CELL R0 — mount, provenance gate, shared helpers
from google.colab import drive
drive.mount("/content/drive")

import os, sys, json, hashlib, platform, gc, time, shutil
import numpy as np, pandas as pd, joblib
import sklearn, scipy, xgboost

BASE = "/content/drive/MyDrive/TrustShift_IDS"
V2   = f"{BASE}/V2_adaptive_reliability"
OUT  = f"{V2}/revision_R1"
os.makedirs(OUT, exist_ok=True)

P = {
  "model_b":      f"{BASE}/models/xgboost/XGB_binary_seed42.joblib",
  "model_m":      f"{BASE}/models/xgboost/XGB_multiclass_seed42.joblib",
  "policy_json":  f"{V2}/adaptive_policy/V2_STEP_10B_FROZEN_PRIMARY_RELIABILITY_POLICY.json",
  "cal":          f"{BASE}/data/processed/CICIDS2017_68F_imputed/CICIDS2017_calibration_68F_imputed.parquet",
  "iid":          f"{BASE}/data/processed/CICIDS2017_68F_imputed/CICIDS2017_iid_test_68F_imputed.parquet",
  "s3_protocol":  f"{V2}/controlled_shifts/S3_corruption/V2_STEP_6D_S3_CORRUPTION_REALIZATION.json",
  "medians":      f"{V2}/controlled_shifts/Level2_compounds/reconstruction_reference/V2_FULL_TRAINING_FEATURE_MEDIANS_68F.npy",
  "s4_recon":     f"{V2}/artifact_audits/V2_STEP_7B_S4_IMPORTANT_EXACT_RECONSTRUCTION.csv",
  "s3_monitor":   f"{V2}/controlled_shifts/S3_corruption/V2_S3_PAIRED_PRIMARY_SSS_ALL_CONDITIONS.csv",
  "s4_monitor":   f"{V2}/controlled_shifts/S4_missing_features/V2_S4_IMPORTANT_PRIMARY_SSS_windows.csv",
  "s3_frozen_out":f"{V2}/adaptive_policy/primary_evaluation/S3/V2_STEP_11B_S3_PRIMARY_ADAPTIVE_WINDOW_RESULTS.csv",
  "s4_frozen_out":f"{V2}/adaptive_policy/primary_evaluation/S4/V2_STEP_11E_S4_PRIMARY_ADAPTIVE_WINDOW_RESULTS.csv",
  "ms_monitor":   f"{V2}/robustness_evaluation/multiseed_streaming/V2_STEP_13C_MULTI_SEED_LABEL_FREE_MONITORING_WINDOWS.csv",
  "ms_frozen_out":f"{V2}/robustness_evaluation/multiseed_streaming/V2_STEP_13D_MULTI_SEED_FROZEN_POLICY_WINDOW_OUTCOMES.csv",
  "s1_monitor":   f"{V2}/deployment_evaluation/V2_S1_EXTERNAL_PRIMARY_SSS_windows.csv",
  "s1_frozen_out":f"{V2}/deployment_evaluation/V2_S1_EXTERNAL_SSS_RELIABILITY_ASSOCIATION.csv",
  "s1_pred_dir":  f"{BASE}/results/shift_evaluation/S1_external/predictions",
  "null_sss":     f"{V2}/shift_monitoring/V2_source_null_PRIMARY_SSS.csv",
  "norm_params":  f"{V2}/shift_monitoring/V2_monitoring_normalization_parameters.csv",
  "ablation_win": f"{V2}/ablation_evaluation/primary_monitor_family_ablation/V2_STEP_14B_LABEL_FREE_FAMILY_ABLATION_WINDOWS.csv",
  "ablation_out": f"{V2}/ablation_evaluation/primary_monitor_family_ablation/V2_STEP_14C_FAMILY_ABLATION_WINDOW_OUTCOMES.csv",
  "l2_out":       f"{V2}/adaptive_policy/primary_evaluation/Level2_compounds/V2_STEP_11F_C_LEVEL2_FROZEN_ADAPTIVE_WINDOW_RESULTS.csv",
  "loao_fixed":   f"{BASE}/results/shift_evaluation/S2_LOAO/Step_12_3E_LOAO_unseen_fixed_coverage.csv",
}
for s in [1, 7, 21, 42, 84]:
    P[f"ms_X_{s}"] = f"{V2}/robustness_evaluation/multiseed_streaming/V2_STEP_13B_SEED_{s}_FLOAT64_STREAM_FEATURES.npz"

def sha256(path, block=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(block), b""):
            h.update(b)
    return h.hexdigest()

missing = [k for k, p in P.items() if not os.path.exists(p)]
print("Missing artifacts:", missing if missing else "none")
if missing:
    raise FileNotFoundError("Paste this output back; do not continue.")

manifest = {"timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
            "python": sys.version.split()[0], "platform": platform.platform(),
            "numpy": np.__version__, "pandas": pd.__version__, "sklearn": sklearn.__version__,
            "scipy": scipy.__version__, "xgboost": xgboost.__version__, "artifacts": {}}
for k, p in P.items():
    if os.path.isdir(p):
        continue
    sz = os.path.getsize(p)
    manifest["artifacts"][k] = {"path": p, "bytes": sz, "sha256": sha256(p) if sz < 3e9 else None}

if xgboost.__version__ != "3.4.1":
    print(f"WARNING: frozen runs used XGBoost 3.4.1; this runtime has {xgboost.__version__}. "
          "The regression gates below will tell us whether predictions still match.")

# ---- frozen constants (manuscript Section 3.4 / 3.8) -------------------------------------
T_B, T_M = 0.9744155346869672, 1.1709901909513676
EPS = 0.01
WINDOW = 5000
THR = {"normal":   {"binary": 0.49778103828430176, "multiclass": 0.49221134185791016},
       "elevated": {"binary": 0.0012131255763865,  "multiclass": 0.0016204167165527},
       "severe":   {"binary": 4.613128198549532e-05, "multiclass": 8.730944055290735e-05}}
pol = json.load(open(P["policy_json"]))
for st, v in pol["thresholds"].items():
    for task in ["binary", "multiclass"]:
        assert abs(float(v[task]) - THR[st.lower()][task]) < 1e-15, (st, task)
print("[R0] Frozen V2.10B thresholds match manuscript values: PASS")

EXPECTED_MEDIAN_FILE_SHA = "44fa27a24dcfbb9274bbe9f5cdc72aa49069d3566ac34b9dd03dee790ed5d4e7"
assert manifest["artifacts"]["medians"]["sha256"] == EXPECTED_MEDIAN_FILE_SHA, "median SHA mismatch"
training_median = np.load(P["medians"], allow_pickle=False)
print("[R0] Frozen training-median vector SHA: PASS")

FEATURE_ORDER = list(json.load(open(P["s3_protocol"]))["feature_order"])
_constants = {"F31", "F33", "F55", "F56", "F57", "F58", "F59", "F60"}
_expected = [f"F{i:02d}" for i in range(1, 77) if f"F{i:02d}" not in _constants]
assert FEATURE_ORDER == _expected and len(FEATURE_ORDER) == 68
print("[R0] 68-feature order: PASS")

FAMILY_MAP = {"Benign": 0, "DDoS": 1, "DoS": 2, "Brute Force": 3, "Web Attack": 4,
              "Botnet": 5, "Infiltration": 6, "Port Scan": 7, "Heartbleed": 8}

model_b = joblib.load(P["model_b"]); model_m = joblib.load(P["model_m"])

def temperature_scale(probs, T):
    p = np.clip(np.asarray(probs, dtype=np.float64), 1e-15, 1.0)
    z = np.log(p) / T
    z -= z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

def predict(X):
    pb = temperature_scale(model_b.predict_proba(X), T_B)
    pm = temperature_scale(model_m.predict_proba(X), T_M)
    return {"binary": (1.0 - pb.max(1), pb.argmax(1).astype(np.int64)),
            "multiclass": (1.0 - pm.max(1), pm.argmax(1).astype(np.int64))}

def labels_from_frame(df):
    y_m = df["attack_family"].map(FAMILY_MAP)
    assert not y_m.isna().any(), "unmapped attack family"
    y_m = y_m.to_numpy(np.int64)
    raw = df["binary_label"]
    if np.issubdtype(raw.dtype, np.number):
        y_b = raw.to_numpy(np.int64)
    else:
        y_b = raw.astype(str).str.strip().str.lower().map({"benign": 0, "attack": 1, "0": 0, "1": 1}).to_numpy(np.int64)
    assert np.array_equal(y_b, (y_m != 0).astype(np.int64))
    return {"binary": y_b, "multiclass": y_m}

def window_record(risk, pred, y, thr):
    """Counts for one window/threshold. 'Attack' = true non-benign; missed attack = attack predicted benign (class 0)."""
    acc = risk <= thr
    err = pred != y
    att = y != 0
    return {"n": int(len(y)), "acc_n": int(acc.sum()), "acc_err": int((acc & err).sum()),
            "n_err": int(err.sum()), "n_attack": int(att.sum()),
            "acc_attack": int((acc & att).sum()),
            "acc_missed_attack": int((acc & att & (pred == 0)).sum()),
            "acc_benign": int((acc & ~att).sum()),
            "acc_false_alarm": int((acc & ~att & (pred != 0)).sum())}

def max_abs_diff(a, b):
    a = np.asarray(a, dtype=np.float64); b = np.asarray(b, dtype=np.float64)
    both_nan = np.isnan(a) & np.isnan(b)
    d = np.abs(a - b); d[both_nan] = 0.0
    return float(np.nanmax(d)) if d.size else 0.0

json.dump(manifest, open(f"{OUT}/R0_environment_and_artifact_manifest.json", "w"), indent=2)
print("[R0] Manifest saved:", f"{OUT}/R0_environment_and_artifact_manifest.json")
print({k: manifest[k] for k in ["python", "numpy", "pandas", "sklearn", "scipy", "xgboost"]})

### CELL R1 — calibration-side quantities (reproduction + post hoc baselines' source parameters)

In [ ]:
# CELL R1 — calibration-side quantities (reproduction + post hoc baselines' source parameters)
cal = pd.read_parquet(P["cal"])
assert len(cal) == 274787
Xc = cal[FEATURE_ORDER].to_numpy(np.float64)
yc = labels_from_frame(cal)
pc = predict(Xc)
del Xc; gc.collect()

err_counts = {t: int((pc[t][1] != yc[t]).sum()) for t in pc}
print("[R1] Calibration errors:", err_counts, "(frozen: binary 138, multiclass 159)")
assert err_counts == {"binary": 138, "multiclass": 159}, "REPRODUCTION FAILURE: calibration errors differ"

# Reproduce frozen state-threshold coverage on calibration (V2.10A-R / V2.10B)
expected_cov = {("elevated", "binary"): 0.990010, ("elevated", "multiclass"): 0.990000,
                ("severe", "binary"): 0.899999, ("severe", "multiclass"): 0.899999}
for (st, t), ev in expected_cov.items():
    cov = float(np.mean(pc[t][0] <= THR[st][t]))
    print(f"    {st:8s} {t:10s} calibration coverage = {cov:.6f} (frozen {ev:.6f})")
    assert abs(cov - ev) < 2e-6, "REPRODUCTION FAILURE: frozen threshold coverage"
print("[R1] Frozen-threshold calibration coverage: PASS")

# POST HOC: static-threshold grid defined from calibration coverage targets
COV_GRID = [0.999, 0.995, 0.99, 0.98, 0.975, 0.95, 0.925, 0.90, 0.85, 0.80, 0.75, 0.70, 0.60, 0.50]
grid_rows = []
THRESH_SET = {}   # name -> {task: thr}
for st in ["normal", "elevated", "severe"]:
    THRESH_SET[f"state_{st}"] = {t: THR[st][t] for t in ["binary", "multiclass"]}
for c in COV_GRID:
    THRESH_SET[f"cov_{c:.3f}"] = {}
    for t in ["binary", "multiclass"]:
        r = np.sort(pc[t][0]); k = int(np.ceil(c * len(r))) - 1
        thr = float(r[k])
        THRESH_SET[f"cov_{c:.3f}"][t] = thr
for name, d in THRESH_SET.items():
    for t, thr in d.items():
        acc = pc[t][0] <= thr
        grid_rows.append({"thr_name": name, "task": t, "threshold": thr,
                          "cal_coverage": float(acc.mean()),
                          "cal_accepted_errors": int((acc & (pc[t][1] != yc[t])).sum())})
grid = pd.DataFrame(grid_rows)
grid.to_csv(f"{OUT}/R1_threshold_set_calibration.csv", index=False)
print(grid.to_string(index=False))

# POST HOC: ATC-MC (Garg et al., ICLR 2022) source parameters
ATC = {}
for t in ["binary", "multiclass"]:
    conf = 1.0 - pc[t][0]
    src_err = float(np.mean(pc[t][1] != yc[t]))
    thr_conf = float(np.quantile(conf, src_err))
    ATC[t] = {"source_error": src_err, "conf_threshold": thr_conf,
              "source_frac_below": float(np.mean(conf < thr_conf))}
print("[R1] ATC parameters:", json.dumps(ATC, indent=1))

# Source-null windows: identical construction to V2.3A (seed 42 permutation, 54 x 5000)
perm_null = np.random.default_rng(42).permutation(len(cal))[:54 * WINDOW].reshape(54, WINDOW)
null_rows = []
for w in range(54):
    idx = perm_null[w]
    rest = np.ones(len(cal), bool); rest[idx] = False
    rec = {"window_id": w}
    for t in ["binary", "multiclass"]:
        conf = 1.0 - pc[t][0]
        # leave-window-out ATC threshold (mirrors the leave-window-out logic of the source null)
        e_rest = float(np.mean(pc[t][1][rest] != yc[t][rest]))
        thr_lwo = float(np.quantile(conf[rest], e_rest))
        rec[f"{t}_atc_est"] = float(np.mean(conf[idx] < thr_lwo))
        rec[f"{t}_err_rate"] = float(np.mean(pc[t][1][idx] != yc[t][idx]))
        for st in ["normal", "elevated", "severe"]:
            r = window_record(pc[t][0][idx], pc[t][1][idx], yc[t][idx], THR[st][t])
            rec[f"{t}_{st}_coverage"] = r["acc_n"] / r["n"]
            rec[f"{t}_{st}_acc_err"] = r["acc_err"]
    null_rows.append(rec)
null_df = pd.DataFrame(null_rows)
null_df.to_csv(f"{OUT}/R1_source_null_windows_ATC_and_outcomes.csv", index=False)
json.dump({"ATC": ATC, "COV_GRID": COV_GRID,
           "THRESH_SET": THRESH_SET, "posthoc": True}, open(f"{OUT}/R1_parameters.json", "w"), indent=2)
print(null_df.describe().T[["mean", "50%", "max"]].to_string())
print("[R1] COMPLETE")

### CELL R2 — multi-seed deployment streams: reproduction + per-window outcomes for every threshold

In [ ]:
# CELL R2 — multi-seed deployment streams: reproduction + per-window outcomes for every threshold
ms_mon = pd.read_csv(P["ms_monitor"]).sort_values(["deployment_order_seed", "global_window_id"])
ms_frozen = pd.read_csv(P["ms_frozen_out"]).sort_values(["deployment_order_seed", "global_window_id"]).reset_index(drop=True)
iid_lab = labels_from_frame(pd.read_parquet(P["iid"], columns=["binary_label", "attack_family"]))

rows, repro = [], []
for seed in [1, 7, 21, 42, 84]:
    z = np.load(P[f"ms_X_{seed}"], allow_pickle=False)
    X = np.asarray(z["X"], np.float64); src = np.asarray(z["source_row_index"], np.int64)
    assert z["feature_order"].tolist() == FEATURE_ORDER and X.shape == (300000, 68)
    z.close()
    pr = predict(X); del X
    y = {t: iid_lab[t][src] for t in iid_lab}
    mon = ms_mon[ms_mon.deployment_order_seed == seed].reset_index(drop=True)
    assert len(mon) == 60
    for w in range(60):
        a, b = w * WINDOW, (w + 1) * WINDOW
        m = mon.iloc[w]; state = str(m["primary_shift_evidence_state"]).lower()
        assert int(m["global_window_id"]) == w
        base = {"dataset": "multiseed_stream", "seed": seed, "window_id": w,
                "stage_name": m["stage_name"], "SSS": float(m["SSS_primary"]), "state": state}
        for t in ["binary", "multiclass"]:
            risk, pred = pr[t][0][a:b], pr[t][1][a:b]
            conf = 1.0 - risk
            atc = float(np.mean(conf < ATC[t]["conf_threshold"]))
            for name, thrs in THRESH_SET.items():
                r = window_record(risk, pred, y[t][a:b], thrs[t])
                rows.append({**base, "task": t, "thr_name": name, "threshold": thrs[t], "atc_est": atc, **r})
            # reproduction record (fixed = normal; adaptive = frozen state threshold)
            rf = window_record(risk, pred, y[t][a:b], THR["normal"][t])
            ra = window_record(risk, pred, y[t][a:b], THR[state][t])
            repro.append({"seed": seed, "global_window_id": w, "task": t,
                          "fixed_cov": rf["acc_n"] / rf["n"], "fixed_risk": rf["acc_err"] / rf["acc_n"] if rf["acc_n"] else np.nan,
                          "adapt_cov": ra["acc_n"] / ra["n"], "adapt_risk": ra["acc_err"] / ra["acc_n"] if ra["acc_n"] else np.nan})
    del pr; gc.collect()
    print(f"  seed {seed}: done")

ms_out = pd.DataFrame(rows)
rp = pd.DataFrame(repro)
worst = 0.0
for t in ["binary", "multiclass"]:
    r = rp[rp.task == t].sort_values(["seed", "global_window_id"]).reset_index(drop=True)
    for mine, frozen in [("fixed_cov", f"{t}_fixed_coverage"), ("fixed_risk", f"{t}_fixed_risk"),
                         ("adapt_cov", f"{t}_adaptive_coverage"), ("adapt_risk", f"{t}_adaptive_risk")]:
        d = max_abs_diff(r[mine], ms_frozen[frozen]); worst = max(worst, d)
        print(f"    {t:10s} {mine:10s} max|diff| vs frozen V2.13D = {d:.3e}")
print("[R2] REPRODUCTION vs frozen V2.13D:", "PASS" if worst < 1e-12 else f"FAIL (max diff {worst:.3e})")
ms_out.to_csv(f"{OUT}/R2_multiseed_window_threshold_outcomes.csv.gz", index=False)
print("[R2] rows:", len(ms_out), "| saved R2_multiseed_window_threshold_outcomes.csv.gz")

### CELL R3 — S3 (82 paired windows) and S4: reproduction + per-window outcomes for every threshold

In [ ]:
# CELL R3 — S3 (82 paired windows) and S4: reproduction + per-window outcomes for every threshold
iid = pd.read_parquet(P["iid"])
assert len(iid) == 412181
X_clean = iid[FEATURE_ORDER].to_numpy(np.float64)
y_iid = labels_from_frame(iid); del iid; gc.collect()
used_idx = np.random.default_rng(42).permutation(len(X_clean))[:82 * WINDOW]

s3_mon = pd.read_csv(P["s3_monitor"])
s3_mon["state_norm"] = s3_mon["primary_shift_evidence_state"].astype(str).str.lower()
s4_mon = pd.read_csv(P["s4_monitor"]); s4_mon["state"] = s4_mon["state"].astype(str).str.lower()
recon = pd.read_csv(P["s4_recon"])
s4_sets = {round(float(r["severity"]), 2): [f.strip() for f in str(r["selected_features"]).split("|") if f.strip()]
           for _, r in recon.iterrows()}

conditions = [("S3", "clean", None), ("S3", "S3_05", "05"), ("S3", "S3_10", "10"), ("S3", "S3_20", "20"), ("S3", "S3_30", "30"),
              ("S4", "important_missing_05", 0.05), ("S4", "important_missing_10", 0.10),
              ("S4", "important_missing_20", 0.20), ("S4", "important_missing_30", 0.30)]
rows, repro = [], []
for fam, cond, spec in conditions:
    if fam == "S3" and spec is None:
        X = X_clean
    elif fam == "S3":
        mask = np.load(f"{V2}/controlled_shifts/S3_corruption/V2_S3_corruption_mask_{spec}.npy").astype(np.int64)
        X = X_clean.copy(); X[np.arange(len(X))[:, None], mask] = training_median[mask]; del mask
    else:
        sel = np.array([FEATURE_ORDER.index(f) for f in s4_sets[spec]], np.int64)
        X = X_clean.copy(); X[:, sel] = training_median[sel]
    pr = predict(X)
    if X is not X_clean: del X
    if fam == "S3":
        mon = s3_mon[s3_mon.condition == cond].sort_values("window_id").reset_index(drop=True)
        states = mon["state_norm"].tolist(); sss = mon["SSS_primary"].tolist()
    else:
        mon = s4_mon[s4_mon.condition == cond].sort_values("window").reset_index(drop=True)
        states = mon["state"].tolist(); sss = mon["SSS_primary"].tolist()
    assert len(states) == 82
    for w in range(82):
        idx = used_idx[w * WINDOW:(w + 1) * WINDOW]
        base = {"dataset": fam, "condition": cond, "window_id": w, "SSS": float(sss[w]), "state": states[w]}
        for t in ["binary", "multiclass"]:
            risk, pred, yy = pr[t][0][idx], pr[t][1][idx], y_iid[t][idx]
            atc = float(np.mean((1.0 - risk) < ATC[t]["conf_threshold"]))
            for name, thrs in THRESH_SET.items():
                rows.append({**base, "task": t, "thr_name": name, "threshold": thrs[t], "atc_est": atc,
                             **window_record(risk, pred, yy, thrs[t])})
            rf = window_record(risk, pred, yy, THR["normal"][t]); ra = window_record(risk, pred, yy, THR[states[w]][t])
            repro.append({"family": fam, "condition": cond, "window_id": w, "task": t,
                          "fixed_cov": rf["acc_n"] / rf["n"], "fixed_risk": rf["acc_err"] / rf["acc_n"] if rf["acc_n"] else np.nan,
                          "adapt_cov": ra["acc_n"] / ra["n"], "adapt_risk": ra["acc_err"] / ra["acc_n"] if ra["acc_n"] else np.nan})
    del pr; gc.collect()
    print(f"  {cond}: done")

ctl = pd.DataFrame(rows); rp = pd.DataFrame(repro)
f3 = pd.read_csv(P["s3_frozen_out"]); f4 = pd.read_csv(P["s4_frozen_out"])
worst = 0.0
for fam, fz in [("S3", f3), ("S4", f4)]:
    for cond in rp[rp.family == fam].condition.unique():
        if fam == "S4" and cond not in set(fz.condition):
            continue
        for t in ["binary", "multiclass"]:
            mine = rp[(rp.family == fam) & (rp.condition == cond) & (rp.task == t)].sort_values("window_id")
            fr = fz[fz.condition == cond].sort_values("window_id")
            for a_, b_ in [("fixed_cov", f"{t}_fixed_coverage"), ("fixed_risk", f"{t}_fixed_risk"),
                           ("adapt_cov", f"{t}_adaptive_coverage"), ("adapt_risk", f"{t}_adaptive_risk")]:
                d = max_abs_diff(mine[a_].to_numpy(), fr[b_].to_numpy()); worst = max(worst, d)
    print(f"    {fam}: compared with frozen window results")
print("[R3] REPRODUCTION vs frozen V2.11B/V2.11E:", "PASS" if worst < 1e-12 else f"FAIL (max diff {worst:.3e})")
ctl.to_csv(f"{OUT}/R3_S3_S4_window_threshold_outcomes.csv.gz", index=False)
print("[R3] rows:", len(ctl))
del X_clean; gc.collect()

### CELL R4 — S1 external (CSE-CIC-IDS2018): reproduction + per-window outcomes for every threshold

In [ ]:
# CELL R4 — S1 external (CSE-CIC-IDS2018): reproduction + per-window outcomes for every threshold
import pyarrow.parquet as pq
shards = sorted(os.path.join(P["s1_pred_dir"], f) for f in os.listdir(P["s1_pred_dir"]) if f.endswith(".parquet"))
assert len(shards) == 10
schema = pq.ParquetFile(shards[0]).schema.names
print("[R4] S1 shard schema:", schema)
need = ["binary_error", "binary_risk", "binary_accept_frozen", "multiclass_error", "multiclass_risk", "multiclass_accept_frozen"]
# optional label/prediction columns for security-specific (missed-attack / false-alarm) metrics
cand = {k: [c for c in schema if k in c.lower()] for k in ["true", "label", "pred", "family"]}
print("[R4] candidate label/pred columns:", cand)

cols = {c: [] for c in need}; shard_id = []
for i, sp in enumerate(shards):
    tb = pq.read_table(sp, columns=need)
    for c in need:
        cols[c].append(tb.column(c).to_numpy())
    shard_id.append(np.full(tb.num_rows, i + 1, np.int16))
cols = {c: np.concatenate(v) for c, v in cols.items()}; shard_id = np.concatenate(shard_id)
N = len(shard_id); assert N == 16_232_943
for t in ["binary", "multiclass"]:
    mism = int(np.sum((cols[f"{t}_risk"] <= THR["normal"][t]) != cols[f"{t}_accept_frozen"].astype(bool)))
    print(f"    {t}: rows where (risk <= frozen Normal threshold) != stored accept flag: {mism}")

s1_mon = pd.read_csv(P["s1_monitor"]).sort_values("window_id").reset_index(drop=True)
n_win = N // WINDOW; assert n_win == 3246 == len(s1_mon)
rows = []
for w in range(n_win):
    a, b = w * WINDOW, (w + 1) * WINDOW
    base = {"dataset": "S1_external", "window_id": w, "first_shard": int(shard_id[a]), "last_shard": int(shard_id[b - 1]),
            "SSS": float(s1_mon.loc[w, "SSS_primary"]), "state": str(s1_mon.loc[w, "primary_shift_evidence_state"]).lower()}
    for t in ["binary", "multiclass"]:
        risk = cols[f"{t}_risk"][a:b]; err = cols[f"{t}_error"][a:b].astype(bool)
        atc = float(np.mean((1.0 - risk) < ATC[t]["conf_threshold"]))
        for name, thrs in THRESH_SET.items():
            acc = risk <= thrs[t]
            rows.append({**base, "task": t, "thr_name": name, "threshold": thrs[t], "atc_est": atc,
                         "n": WINDOW, "acc_n": int(acc.sum()), "acc_err": int((acc & err).sum()), "n_err": int(err.sum())})
s1 = pd.DataFrame(rows)
fz = pd.read_csv(P["s1_frozen_out"]).sort_values("window_id").reset_index(drop=True)
worst = 0.0
for t in ["binary", "multiclass"]:
    d = s1[(s1.task == t) & (s1.thr_name == "state_normal")].sort_values("window_id").reset_index(drop=True)
    worst = max(worst, max_abs_diff(d.acc_n / d.n, fz[f"{t}_coverage_frozen"]),
                max_abs_diff(np.where(d.acc_n > 0, d.acc_err / d.acc_n.clip(lower=1), np.nan), fz[f"{t}_selective_risk_frozen"]))
print("[R4] REPRODUCTION vs frozen V2.5D:", "PASS" if worst < 1e-12 else f"FAIL (max diff {worst:.3e})")
s1.to_csv(f"{OUT}/R4_S1_window_threshold_outcomes.csv.gz", index=False)
print("[R4] Quick look (post hoc) — whole-stream totals by threshold:")
tot = s1.groupby(["task", "thr_name"])[["acc_n", "acc_err", "n"]].sum()
tot["coverage"] = tot.acc_n / tot.n; tot["sel_risk"] = tot.acc_err / tot.acc_n
viol = s1.assign(v=(s1.acc_err / s1.acc_n.clip(lower=1) > EPS) & (s1.acc_n > 0)).groupby(["task", "thr_name"]).v.sum()
tot["violating_windows"] = viol
print(tot.loc[(slice(None), ["state_normal", "state_elevated", "state_severe", "cov_0.700", "cov_0.500"]), :].to_string())
del cols; gc.collect()

### CELL R5 — package everything for upload

In [ ]:
# CELL R5 — package everything for upload
EXP = f"{OUT}/export"; os.makedirs(EXP, exist_ok=True)
copy_keys = ["s3_monitor", "s4_monitor", "ms_monitor", "ms_frozen_out", "s1_monitor", "s1_frozen_out", "null_sss",
             "norm_params", "ablation_win", "ablation_out", "l2_out", "s3_frozen_out", "s4_frozen_out", "loao_fixed",
             "s4_recon", "policy_json"]
for k in copy_keys:
    shutil.copy2(P[k], f"{EXP}/{os.path.basename(P[k])}")
for f in os.listdir(OUT):
    if f.startswith(("R0_", "R1_", "R2_", "R3_", "R4_")):
        shutil.copy2(f"{OUT}/{f}", f"{EXP}/{f}")
zip_path = shutil.make_archive(f"{OUT}/TrustShift_revision_R1_export", "zip", EXP)
print("Upload this file to the chat:", zip_path, f"({os.path.getsize(zip_path)/1e6:.1f} MB)")